In [ ]:
# 1. Установка зависимостей
# !pip install tensorflow==2.18.0 numpy==1.26.4
# !pip install rasterio==1.3.9 earthengine-api==0.1.408
# !pip install google-cloud-storage

import ee
import tensorflow as tf
import numpy as np
import rasterio
import os
import glob
import time
from google.colab import drive
import matplotlib.pyplot as plt

# 2. Инициализация Google Earth Engine
drive.mount('/content/drive')
ee.Authenticate()
ee.Initialize(project='ai-terrain-detection')

# 3. Экспорт данных LIDAR
def export_lidar_data():
    """Экспорт данных с правильными размерами"""
    try:
        region = ee.Geometry.Rectangle([-118.5, 34.1, -117.5, 35.1])  # Новый регион
        collection = ee.ImageCollection('USGS/3DEP/1m') \
                      .filterBounds(region) \
                      .mosaic() \
                      .select('elevation') \
                      .clip(region)

        # Создание LR 42x42 (252/6)
        dem_lr = collection.resample('bilinear').reproject(crs='EPSG:4326', scale=42)

        task = ee.batch.Export.image.toDrive(
            image=dem_lr,
            description='LIDAR_EXPORT',
            folder='LIDAR_DATA',
            fileNamePrefix='lidar_export',
            scale=42,
            region=region,
            maxPixels=1e13,
            fileFormat='GeoTIFF'
        )
        task.start()
        print("Экспорт начат...")
        while task.active():
            time.sleep(30)
        print("Экспорт успешен!")

    except Exception as e:
        print(f"Ошибка: {str(e)}")
        raise

# 4. Класс обработки данных с фиксированными размерами
class DEMProcessor:
    def __init__(self, hr_size=252, scale=6):
        self.hr_size = hr_size
        self.scale = scale
        self.lr_size = hr_size // scale
        self.nodata = -9999

    @tf.function
    def process_tile(self, data):
        mask = tf.math.equal(data, self.nodata)
        valid = tf.boolean_mask(data, ~mask)
        min_val = tf.reduce_min(valid)
        max_val = tf.reduce_max(valid)
        range_val = tf.maximum(max_val - min_val, 1e-6)

        data = tf.where(mask, min_val, data)
        normalized = (data - min_val) / range_val

        hr = tf.image.resize(normalized, [self.hr_size, self.hr_size], 'bicubic')
        lr = tf.image.resize(hr, [self.lr_size, self.lr_size], 'area')

        return (lr, hr)

# 5. Создание Dataset
def create_dataset(file_list):
    processor = DEMProcessor()

    def _parse_file(file_path):
        def _read(path):
            with rasterio.open(path.decode()) as src:
                arr = src.read(1).astype(np.float32)
                return arr[..., np.newaxis]

        data = tf.numpy_function(_read, [file_path], tf.float32)
        data.set_shape([None, None, 1])
        return processor.process_tile(data)

    return tf.data.Dataset.from_tensor_slices(file_list) \
           .map(_parse_file, num_parallel_calls=tf.data.AUTOTUNE) \
           .batch(32) \
           .prefetch(tf.data.AUTOTUNE)

# 6. Модель EDSR с точными размерами
class EDSR(tf.keras.Model):
    def __init__(self, scale=6):
        super().__init__()
        self.scale = scale

        self.conv1 = tf.keras.layers.Conv2D(256, 3, padding='same')
        self.res_blocks = [self.ResBlock() for _ in range(8)]

        self.upscale = tf.keras.Sequential([
            tf.keras.layers.Conv2D(256 * (scale**2), 3, padding='same'),
            tf.keras.layers.Lambda(lambda x: tf.nn.depth_to_space(x, scale)),
            tf.keras.layers.Conv2D(1, 3, padding='same')
        ])

    class ResBlock(tf.keras.layers.Layer):
        def __init__(self):
            super().__init__()
            self.conv1 = tf.keras.layers.Conv2D(256, 3, padding='same', activation='relu')
            self.conv2 = tf.keras.layers.Conv2D(256, 3, padding='same')

        def call(self, x):
            return x + self.conv2(self.conv1(x))

    def call(self, inputs):
        x = self.conv1(inputs)
        for block in self.res_blocks:
            x = block(x)
        return self.upscale(x)

# 7. Конфигурация обучения
train_config = {
    'hr_size': 252,
    'scale': 6,
    'epochs': 100,
    'batch_size': 32,
    'learning_rate': 1e-4
}

# 8. Обучение модели
def main():
    # export_lidar_data()  # Раскомментировать для первого запуска

    files = glob.glob('/content/drive/MyDrive/LIDAR_DATA/*.tif')
    dataset = create_dataset(files)

    # Проверка размеров
    for lr, hr in dataset.take(1):
        print(f"LR shape: {lr.shape}, HR shape: {hr.shape}")  # Должно быть (32,42,42,1) и (32,252,252,1)

    model = EDSR(scale=train_config['scale'])
    model.compile(optimizer=tf.keras.optimizers.Adam(train_config['learning_rate']),
                  loss='mse')

    history = model.fit(
        dataset,
        epochs=train_config['epochs'],
        callbacks=[
            tf.keras.callbacks.ModelCheckpoint('edsr_model.h5', save_best_only=True),
            tf.keras.callbacks.TensorBoard()
        ]
    )

    # Тестирование
    test_input = tf.random.normal([1,42,42,1])
    prediction = model.predict(test_input)
    final_model_path = os.path.join('/content/drive/MyDrive/', 'dem_sr_final_2.weights.h5')
    model.save_weights(final_model_path)
    print(f"Output shape: {prediction.shape}")  # Должно быть (1,252,252,1)

if __name__ == "__main__":
    main()

In [ ]:
!pip install tensorflow==2.18.0 numpy==1.26.4
!pip install rasterio==1.3.9 earthengine-api==0.1.408
!pip install google-cloud-storage